In [1]:
#!/usr/bin/env python
# coding: utf-8

"""
3-CANDLE PATTERN ANALYSIS: CONTINUATIONS AND RETRACEMENTS - FIXED VERSION
Analyzes candle 3 behavior based on candle 2's relationship to candle 1
Tracks multiple timeframes: 15m, 30m, 1h, 4h, 8h, 1D

FIXES:
1. Corrected percentage calculations in filter exports
2. Converted ranges from points to percentages
"""

import pandas as pd
import numpy as np
from collections import defaultdict
from datetime import time

# Load the data
file_path = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
df = pd.read_csv(file_path)

# Combine Date and Time into a datetime column
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.set_index('Datetime', inplace=True)

# Filter data from 2013 to 2025
start_date = pd.to_datetime('2013-01-01')
end_date = pd.to_datetime('2025-12-31')
df = df.loc[start_date:end_date]

print("=" * 120)
print("3-CANDLE PATTERN ANALYSIS: CONTINUATIONS & RETRACEMENTS - FIXED VERSION")
print("=" * 120)
print(f"\nData loaded: {len(df)} rows from {df.index[0]} to {df.index[-1]}")

# Define timeframes to analyze
TIMEFRAMES = {
    '15m': '15min',
    '30m': '30min', 
    '1h': '1H',
    '4h': '4H',
    '8h': '8H',
    '1D': '1D'
}

# Define filter categories
STRENGTH_BUCKETS = [0, 0.1, 0.25, 0.5, 1.0, 2.0, float('inf')]  # % beyond C1 high/low
STRENGTH_LABELS = ['0-0.1%', '0.1-0.25%', '0.25-0.5%', '0.5-1.0%', '1.0-2.0%', '2.0%+']

def resample_to_timeframe(df, timeframe):
    """Resample 1-minute data to specified timeframe"""
    resampled = df.resample(timeframe).agg({
        'Open': 'first',
        'High': 'max',
        'Low': 'min',
        'Close': 'last',
        'Volume': 'sum'
    }).dropna()
    return resampled

def calculate_candle_levels(candle):
    """Calculate key levels for a candle"""
    high = candle['High']
    low = candle['Low']
    open_price = candle['Open']
    close = candle['Close']
    
    range_size = high - low
    
    levels = {
        'high': high,
        'low': low,
        'open': open_price,
        'close': close,
        'range': range_size,
        '75_pct': low + (range_size * 0.75),
        '50_pct': low + (range_size * 0.50),
        '25_pct': low + (range_size * 0.25)
    }
    
    return levels

def classify_candle2_vs_candle1(c1_levels, c2_levels):
    """
    Classify candle 2's close relative to candle 1
    Returns: 'above_high', 'below_low', 'inside_range'
    """
    c2_close = c2_levels['close']
    c1_high = c1_levels['high']
    c1_low = c1_levels['low']
    
    if c2_close > c1_high:
        return 'above_high'
    elif c2_close < c1_low:
        return 'below_low'
    else:
        return 'inside_range'

def analyze_candle3_vs_candle2(c2_levels, c3_levels):
    """
    Analyze where candle 3 closes relative to all candle 2 levels
    Returns dict of boolean flags
    """
    c3_close = c3_levels['close']
    
    results = {
        'above_c2_high': c3_close > c2_levels['high'],
        'below_c2_low': c3_close < c2_levels['low'],
        'above_c2_open': c3_close > c2_levels['open'],
        'below_c2_open': c3_close < c2_levels['open'],
        'above_c2_close': c3_close > c2_levels['close'],
        'below_c2_close': c3_close < c2_levels['close'],
        'above_c2_75pct': c3_close > c2_levels['75_pct'],
        'below_c2_75pct': c3_close < c2_levels['75_pct'],
        'above_c2_50pct': c3_close > c2_levels['50_pct'],
        'below_c2_50pct': c3_close < c2_levels['50_pct'],
        'above_c2_25pct': c3_close > c2_levels['25_pct'],
        'below_c2_25pct': c3_close < c2_levels['25_pct'],
    }
    
    return results

def get_time_bucket(timestamp):
    """Categorize time of day into buckets"""
    hour = timestamp.hour
    
    if 18 <= hour <= 23:
        return 'Overnight'
    elif 0 <= hour < 2:
        return 'Overnight'
    elif 2 <= hour < 9:
        return 'PreMarket'
    elif 9 <= hour < 16:
        return 'RTH'  # Regular Trading Hours
    else:
        return 'After_Hours'

def get_day_of_week(timestamp):
    """Get day of week name"""
    days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    return days[timestamp.weekday()]

def get_candle_direction(candle):
    """Determine if candle is bullish or bearish"""
    return 'Bullish' if candle['Close'] > candle['Open'] else 'Bearish'

def calculate_breakout_strength_pct(c1_levels, c2_levels, pattern_type):
    """
    Calculate how far C2 closes beyond C1 as a percentage of C1's close price
    Returns percentage value
    """
    c1_close = c1_levels['close']
    c2_close = c2_levels['close']
    
    if pattern_type == 'above_high':
        c1_reference = c1_levels['high']
        distance = c2_close - c1_reference
    elif pattern_type == 'below_low':
        c1_reference = c1_levels['low']
        distance = c1_reference - c2_close  # Positive value for distance below
    else:
        return 0  # Inside range has no "strength"
    
    # Calculate as percentage of C1's close price
    strength_pct = (distance / c1_close) * 100
    return strength_pct

def get_strength_bucket(strength_pct):
    """Categorize strength into buckets"""
    for i in range(len(STRENGTH_BUCKETS) - 1):
        if strength_pct >= STRENGTH_BUCKETS[i] and strength_pct < STRENGTH_BUCKETS[i+1]:
            return STRENGTH_LABELS[i]
    return STRENGTH_LABELS[-1]

def analyze_timeframe(df_resampled, timeframe_name):
    """
    Analyze 3-candle patterns for a specific timeframe
    """
    print(f"\nAnalyzing {timeframe_name} timeframe...")
    
    # Initialize statistics storage with filters
    def create_stats_dict():
        return {
            'count': 0,
            'c3_outcomes': defaultdict(int),
            'c1_ranges': [],
            'c2_ranges': [],
            'c3_ranges': [],
            'c1_closes': [],  # NEW: Store closes for percentage calculation
            'time_buckets': defaultdict(int),
            # New filter breakdowns - FIXED: Store total counts separately
            'by_c2_direction': {
                'Bullish': {'count': 0, 'outcomes': defaultdict(int)},
                'Bearish': {'count': 0, 'outcomes': defaultdict(int)}
            },
            'by_strength': {
                label: {'count': 0, 'outcomes': defaultdict(int)} 
                for label in STRENGTH_LABELS
            },
            'by_time_session': {
                session: {'count': 0, 'outcomes': defaultdict(int)}
                for session in ['RTH', 'PreMarket', 'Overnight', 'After_Hours']
            },
            'by_day_of_week': {
                day: {'count': 0, 'outcomes': defaultdict(int)}
                for day in ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
            }
        }
    
    stats = {
        'above_high': create_stats_dict(),
        'below_low': create_stats_dict(),
        'inside_range': create_stats_dict()
    }
    
    # Analyze consecutive 3-candle sequences
    for i in range(len(df_resampled) - 2):
        candle1 = df_resampled.iloc[i]
        candle2 = df_resampled.iloc[i + 1]
        candle3 = df_resampled.iloc[i + 2]
        
        # Skip weekends
        if candle2.name.weekday() >= 5:
            continue
        
        # Skip if any candle has zero range (data issues or non-trading periods)
        if candle1['High'] == candle1['Low'] or candle2['High'] == candle2['Low'] or candle3['High'] == candle3['Low']:
            continue
        
        # Calculate levels for each candle
        c1_levels = calculate_candle_levels(candle1)
        c2_levels = calculate_candle_levels(candle2)
        c3_levels = calculate_candle_levels(candle3)
        
        # Classify candle 2 vs candle 1
        c2_classification = classify_candle2_vs_candle1(c1_levels, c2_levels)
        
        # Analyze candle 3 vs candle 2
        c3_analysis = analyze_candle3_vs_candle2(c2_levels, c3_levels)
        
        # Get filter attributes
        time_bucket = get_time_bucket(candle2.name)
        day_of_week = get_day_of_week(candle2.name)
        c1_direction = get_candle_direction(candle1) 
        c2_direction = get_candle_direction(candle2) 
        strength_pct = calculate_breakout_strength_pct(c1_levels, c2_levels, c2_classification)
        strength_bucket = get_strength_bucket(strength_pct) if c2_classification != 'inside_range' else None
        
        # Update statistics
        pattern_stats = stats[c2_classification]
        pattern_stats['count'] += 1
        pattern_stats['c1_ranges'].append(c1_levels['range'])
        pattern_stats['c2_ranges'].append(c2_levels['range'])
        pattern_stats['c3_ranges'].append(c3_levels['range'])
        pattern_stats['c1_closes'].append(c1_levels['close'])  # NEW
        pattern_stats['time_buckets'][time_bucket] += 1
        
        # FIXED: Increment counts separately from outcomes
        pattern_stats['by_c2_direction'][c2_direction]['count'] += 1
        pattern_stats['by_time_session'][time_bucket]['count'] += 1
        pattern_stats['by_day_of_week'][day_of_week]['count'] += 1
        if strength_bucket:
            pattern_stats['by_strength'][strength_bucket]['count'] += 1
        
        # Record all candle 3 outcomes
        for outcome, occurred in c3_analysis.items():
            if occurred:
                pattern_stats['c3_outcomes'][outcome] += 1
                
                # FIXED: Track outcomes correctly
                pattern_stats['by_c2_direction'][c2_direction]['outcomes'][outcome] += 1
                pattern_stats['by_time_session'][time_bucket]['outcomes'][outcome] += 1
                pattern_stats['by_day_of_week'][day_of_week]['outcomes'][outcome] += 1
                
                if strength_bucket:
                    pattern_stats['by_strength'][strength_bucket]['outcomes'][outcome] += 1
    
    return stats

def display_results(all_timeframe_stats):
    """Display comprehensive results across all timeframes"""
    
    print("\n" + "=" * 120)
    print("RESULTS: 3-CANDLE PATTERN ANALYSIS - OVERALL STATISTICS")
    print("=" * 120)
    
    for tf_name, stats in all_timeframe_stats.items():
        print(f"\n{'=' * 120}")
        print(f"TIMEFRAME: {tf_name}")
        print(f"{'=' * 120}")
        
        for pattern_type in ['above_high', 'below_low', 'inside_range']:
            pattern_stats = stats[pattern_type]
            
            if pattern_stats['count'] == 0:
                continue
            
            print(f"\n{'-' * 120}")
            if pattern_type == 'above_high':
                print(f"PATTERN: Candle 2 closes ABOVE Candle 1 high (Bullish Breakout)")
            elif pattern_type == 'below_low':
                print(f"PATTERN: Candle 2 closes BELOW Candle 1 low (Bearish Breakout)")
            else:
                print(f"PATTERN: Candle 2 closes INSIDE Candle 1 range (No Breakout)")
            print(f"{'-' * 120}")
            
            total_count = pattern_stats['count']
            print(f"\nTotal occurrences: {total_count:,}")
            
            # FIXED: Average ranges as percentages
            avg_c1_close = np.mean(pattern_stats['c1_closes'])
            avg_c1_range = np.mean(pattern_stats['c1_ranges'])
            avg_c2_range = np.mean(pattern_stats['c2_ranges'])
            avg_c3_range = np.mean(pattern_stats['c3_ranges'])
            
            avg_c1_range_pct = (avg_c1_range / avg_c1_close) * 100
            avg_c2_range_pct = (avg_c2_range / avg_c1_close) * 100
            avg_c3_range_pct = (avg_c3_range / avg_c1_close) * 100
            
            print(f"\nAverage Candle Ranges (as % of price):")
            print(f"  Candle 1: {avg_c1_range_pct:.3f}%")
            print(f"  Candle 2: {avg_c2_range_pct:.3f}%")
            print(f"  Candle 3: {avg_c3_range_pct:.3f}%")
            
            # Time distribution
            print(f"\nTime of Day Distribution:")
            sorted_times = sorted(pattern_stats['time_buckets'].items(), key=lambda x: x[1], reverse=True)
            for time_bucket, count in sorted_times:
                pct = (count / total_count) * 100
                print(f"  {time_bucket:20s}: {count:5,} ({pct:5.1f}%)")
            
            # Candle 3 outcomes
            print(f"\nCandle 3 Outcomes (% of time each level was reached):")
            print(f"{'Level':30s} {'Count':>10s} {'Percentage':>12s} {'Bar Chart':s}")
            print("-" * 100)
            
            outcomes_sorted = sorted(pattern_stats['c3_outcomes'].items(), key=lambda x: x[1], reverse=True)
            
            for outcome, count in outcomes_sorted:
                pct = (count / total_count) * 100
                bar_length = int(pct / 2)
                bar = '█' * bar_length
                
                # Format outcome name for readability
                outcome_display = outcome.replace('_', ' ').replace('c2', 'C2').replace('pct', '%').title()
                
                print(f"{outcome_display:30s} {count:10,} {pct:11.2f}% {bar}")
            
            # Key continuation/reversal metrics
            print(f"\nKey Metrics:")
            
            if pattern_type == 'above_high':  # Bullish breakout
                continuation = pattern_stats['c3_outcomes'].get('above_c2_high', 0)
                reversal = pattern_stats['c3_outcomes'].get('below_c2_low', 0)
                print(f"  Strong Continuation (C3 > C2 high): {continuation:,} ({continuation/total_count*100:.1f}%)")
                print(f"  Full Reversal (C3 < C2 low):        {reversal:,} ({reversal/total_count*100:.1f}%)")
                
            elif pattern_type == 'below_low':  # Bearish breakout
                continuation = pattern_stats['c3_outcomes'].get('below_c2_low', 0)
                reversal = pattern_stats['c3_outcomes'].get('above_c2_high', 0)
                print(f"  Strong Continuation (C3 < C2 low):  {continuation:,} ({continuation/total_count*100:.1f}%)")
                print(f"  Full Reversal (C3 > C2 high):       {reversal:,} ({reversal/total_count*100:.1f}%)")
            
            else:  # Inside range
                breakout_up = pattern_stats['c3_outcomes'].get('above_c2_high', 0)
                breakout_down = pattern_stats['c3_outcomes'].get('below_c2_low', 0)
                print(f"  Breakout Up (C3 > C2 high):   {breakout_up:,} ({breakout_up/total_count*100:.1f}%)")
                print(f"  Breakout Down (C3 < C2 low):  {breakout_down:,} ({breakout_down/total_count*100:.1f}%)")


def export_to_csv(all_timeframe_stats):
    """Export results to CSV for further analysis - FIXED VERSION"""
    
    # Export 1: Overall statistics
    export_data = []
    
    for tf_name, stats in all_timeframe_stats.items():
        for pattern_type in ['above_high', 'below_low', 'inside_range']:
            pattern_stats = stats[pattern_type]
            
            if pattern_stats['count'] == 0:
                continue
            
            total_count = pattern_stats['count']
            
            # FIXED: Calculate percentage ranges
            avg_c1_close = np.mean(pattern_stats['c1_closes'])
            avg_c1_range = np.mean(pattern_stats['c1_ranges'])
            avg_c2_range = np.mean(pattern_stats['c2_ranges'])
            avg_c3_range = np.mean(pattern_stats['c3_ranges'])
            
            avg_c1_range_pct = (avg_c1_range / avg_c1_close) * 100
            avg_c2_range_pct = (avg_c2_range / avg_c1_close) * 100
            avg_c3_range_pct = (avg_c3_range / avg_c1_close) * 100
            
            # Calculate percentages for each outcome
            row = {
                'Timeframe': tf_name,
                'Pattern': pattern_type,
                'Total_Count': total_count,
                'Avg_C1_Range': avg_c1_range_pct,
                'Avg_C2_Range': avg_c2_range_pct,
                'Avg_C3_Range': avg_c3_range_pct,
            }
            
            # Add outcome percentages
            for outcome in ['above_c2_high', 'below_c2_low', 'above_c2_open', 'below_c2_open',
                           'above_c2_close', 'below_c2_close', 'above_c2_75pct', 'below_c2_75pct',
                           'above_c2_50pct', 'below_c2_50pct', 'above_c2_25pct', 'below_c2_25pct']:
                count = pattern_stats['c3_outcomes'].get(outcome, 0)
                pct = (count / total_count) * 100
                row[f'{outcome}_pct'] = pct
                row[f'{outcome}'] = count
            
            export_data.append(row)
    
    export_df = pd.DataFrame(export_data)
    output_path = r"C:\Users\tammm\NQ and ES analysis\three_candle_pattern_analysis.csv"
    export_df.to_csv(output_path, index=False)
    print(f"\n✓ Overall results exported to: {output_path}")
    
    # Export 2: Filter breakdown - C2 Direction - FIXED
    direction_export_data = []
    
    for tf_name, stats in all_timeframe_stats.items():
        for pattern_type in ['above_high', 'below_low', 'inside_range']:
            pattern_stats = stats[pattern_type]
            
            if pattern_stats['count'] == 0:
                continue
            
            for direction in ['Bullish', 'Bearish']:
                dir_data = pattern_stats['by_c2_direction'][direction]
                dir_total = dir_data['count']
                dir_outcomes = dir_data['outcomes']
                
                if dir_total == 0:
                    continue
                
                row = {
                    'Timeframe': tf_name,
                    'Pattern': pattern_type,
                    'C2_Direction': direction,
                    'Total_Count': dir_total
                }
                
                for outcome in ['above_c2_high', 'below_c2_low', 'above_c2_50pct', 'below_c2_50pct']:
                    count = dir_outcomes.get(outcome, 0)
                    pct = (count / dir_total) * 100 if dir_total > 0 else 0
                    row[f'{outcome}_pct'] = pct
                    row[f'{outcome}'] = count
                
                direction_export_data.append(row)
    
    direction_df = pd.DataFrame(direction_export_data)
    direction_output_path = r"C:\Users\tammm\NQ and ES analysis\three_candle_by_c2_direction.csv"
    direction_df.to_csv(direction_output_path, index=False)
    print(f"✓ C2 Direction breakdown exported to: {direction_output_path}")
    
    # Export 3: Filter breakdown - Strength - FIXED
    strength_export_data = []
    
    for tf_name, stats in all_timeframe_stats.items():
        for pattern_type in ['above_high', 'below_low']:  # Only for breakouts
            pattern_stats = stats[pattern_type]
            
            if pattern_stats['count'] == 0:
                continue
            
            for strength_label in STRENGTH_LABELS:
                strength_data = pattern_stats['by_strength'][strength_label]
                strength_total = strength_data['count']
                strength_outcomes = strength_data['outcomes']
                
                if strength_total == 0:
                    continue
                
                row = {
                    'Timeframe': tf_name,
                    'Pattern': pattern_type,
                    'Strength_Bucket': strength_label,
                    'Total_Count': strength_total
                }
                
                for outcome in ['above_c2_high', 'below_c2_low', 'above_c2_50pct', 'below_c2_50pct']:
                    count = strength_outcomes.get(outcome, 0)
                    pct = (count / strength_total) * 100 if strength_total > 0 else 0
                    row[f'{outcome}_pct'] = pct
                    row[f'{outcome}'] = count
                
                strength_export_data.append(row)
    
    strength_df = pd.DataFrame(strength_export_data)
    strength_output_path = r"C:\Users\tammm\NQ and ES analysis\three_candle_by_strength.csv"
    strength_df.to_csv(strength_output_path, index=False)
    print(f"✓ Breakout Strength breakdown exported to: {strength_output_path}")
    
    # Export 4: Filter breakdown - Time Session - FIXED
    session_export_data = []
    
    for tf_name, stats in all_timeframe_stats.items():
        for pattern_type in ['above_high', 'below_low', 'inside_range']:
            pattern_stats = stats[pattern_type]
            
            if pattern_stats['count'] == 0:
                continue
            
            for session in ['RTH', 'PreMarket', 'Overnight', 'After_Hours']:
                session_data = pattern_stats['by_time_session'][session]
                session_total = session_data['count']
                session_outcomes = session_data['outcomes']
                
                if session_total == 0:
                    continue
                
                row = {
                    'Timeframe': tf_name,
                    'Pattern': pattern_type,
                    'Time_Session': session,
                    'Total_Count': session_total
                }
                
                for outcome in ['above_c2_high', 'below_c2_low', 'above_c2_50pct', 'below_c2_50pct']:
                    count = session_outcomes.get(outcome, 0)
                    pct = (count / session_total) * 100 if session_total > 0 else 0
                    row[f'{outcome}_pct'] = pct
                    row[f'{outcome}'] = count
                
                session_export_data.append(row)
    
    session_df = pd.DataFrame(session_export_data)
    session_output_path = r"C:\Users\tammm\NQ and ES analysis\three_candle_by_session.csv"
    session_df.to_csv(session_output_path, index=False)
    print(f"✓ Time Session breakdown exported to: {session_output_path}")
    
    # Export 5: Filter breakdown - Day of Week - FIXED
    dow_export_data = []
    
    for tf_name, stats in all_timeframe_stats.items():
        for pattern_type in ['above_high', 'below_low', 'inside_range']:
            pattern_stats = stats[pattern_type]
            
            if pattern_stats['count'] == 0:
                continue
            
            for day in ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']:
                day_data = pattern_stats['by_day_of_week'][day]
                day_total = day_data['count']
                day_outcomes = day_data['outcomes']
                
                if day_total == 0:
                    continue
                
                row = {
                    'Timeframe': tf_name,
                    'Pattern': pattern_type,
                    'Day_Of_Week': day,
                    'Total_Count': day_total
                }
                
                for outcome in ['above_c2_high', 'below_c2_low', 'above_c2_50pct', 'below_c2_50pct']:
                    count = day_outcomes.get(outcome, 0)
                    pct = (count / day_total) * 100 if day_total > 0 else 0
                    row[f'{outcome}_pct'] = pct
                    row[f'{outcome}'] = count
                
                dow_export_data.append(row)
    
    dow_df = pd.DataFrame(dow_export_data)
    dow_output_path = r"C:\Users\tammm\NQ and ES analysis\three_candle_by_day_of_week.csv"
    dow_df.to_csv(dow_output_path, index=False)
    print(f"✓ Day of Week breakdown exported to: {dow_output_path}")

# Main execution
print("\n" + "=" * 120)
print("PROCESSING MULTIPLE TIMEFRAMES")
print("=" * 120)

all_timeframe_stats = {}

for tf_name, tf_freq in TIMEFRAMES.items():
    df_resampled = resample_to_timeframe(df, tf_freq)
    print(f"\n{tf_name}: {len(df_resampled):,} candles created")
    
    stats = analyze_timeframe(df_resampled, tf_name)
    all_timeframe_stats[tf_name] = stats

# Display results
display_results(all_timeframe_stats)

# Export to CSV
print("\n" + "=" * 120)
print("EXPORTING RESULTS - FIXED VERSION")
print("=" * 120)
export_to_csv(all_timeframe_stats)

print("\n" + "=" * 120)
print("ANALYSIS COMPLETE")
print("=" * 120)
print(f"\nTimeframes analyzed: {', '.join(TIMEFRAMES.keys())}")
print(f"Pattern types: Candle 2 above C1 high, below C1 low, inside C1 range")
print(f"Candle 3 levels tracked: high, low, open, close, 75%, 50%, 25%")
print(f"\nFIXES APPLIED:")
print(f"  ✓ Percentages now calculated correctly in all exports")
print(f"  ✓ Ranges converted to percentages (% of price)")

3-CANDLE PATTERN ANALYSIS: CONTINUATIONS & RETRACEMENTS - FIXED VERSION

Data loaded: 4161727 rows from 2013-01-02 06:00:00 to 2025-03-14 16:59:00

PROCESSING MULTIPLE TIMEFRAMES

15m: 285,181 candles created

Analyzing 15m timeframe...

30m: 143,991 candles created

Analyzing 30m timeframe...

1h: 72,371 candles created

Analyzing 1h timeframe...

4h: 19,416 candles created

Analyzing 4h timeframe...

8h: 10,027 candles created

Analyzing 8h timeframe...

1D: 3,783 candles created

Analyzing 1D timeframe...

RESULTS: 3-CANDLE PATTERN ANALYSIS - OVERALL STATISTICS

TIMEFRAME: 15m

------------------------------------------------------------------------------------------------------------------------
PATTERN: Candle 2 closes ABOVE Candle 1 high (Bullish Breakout)
------------------------------------------------------------------------------------------------------------------------

Total occurrences: 65,351

Average Candle Ranges (as % of price):
  Candle 1: 0.151%
  Candle 2: 0.185%
 


✓ Overall results exported to: C:\Users\tammm\NQ and ES analysis\three_candle_pattern_analysis.csv
✓ C2 Direction breakdown exported to: C:\Users\tammm\NQ and ES analysis\three_candle_by_c2_direction.csv
✓ Breakout Strength breakdown exported to: C:\Users\tammm\NQ and ES analysis\three_candle_by_strength.csv
✓ Time Session breakdown exported to: C:\Users\tammm\NQ and ES analysis\three_candle_by_session.csv
✓ Day of Week breakdown exported to: C:\Users\tammm\NQ and ES analysis\three_candle_by_day_of_week.csv

ANALYSIS COMPLETE

Timeframes analyzed: 15m, 30m, 1h, 4h, 8h, 1D
Pattern types: Candle 2 above C1 high, below C1 low, inside C1 range
Candle 3 levels tracked: high, low, open, close, 75%, 50%, 25%

FIXES APPLIED:
  ✓ Percentages now calculated correctly in all exports
  ✓ Ranges converted to percentages (% of price)


In [2]:
#!/usr/bin/env python
# coding: utf-8

"""
COMPREHENSIVE ONE-PAGE SUMMARIES - MINIMALIST VERSION
One page per pattern showing ALL C3 statistics together
Clean, monochrome, monospace design
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

# Set minimalist monospace style
plt.rcParams['font.family'] = 'monospace'
plt.rcParams['figure.dpi'] = 300
plt.rcParams['axes.linewidth'] = 0.5

print("=" * 120)
print("CREATING MINIMALIST ONE-PAGE SUMMARIES")
print("=" * 120)

# Load the data
base_path = r"C:\Users\tammm\NQ and ES analysis"
overall_df = pd.read_csv(f"{base_path}\\three_candle_pattern_analysis.csv")

# Fix data if needed
def fix_dataframe(df, fix_percentages=True, convert_ranges=False):
    df_fixed = df.copy()
    if fix_percentages:
        pct_cols = [col for col in df.columns if col.endswith('_pct')]
        for col in pct_cols:
            base_col = col.replace('_pct', '')
            if base_col in df.columns and 'Total_Count' in df.columns:
                df_fixed[col] = (df[base_col] / df['Total_Count']) * 100
    if convert_ranges:
        avg_price = 12000
        range_cols = [col for col in df.columns if 'Range' in col and 'Avg' in col]
        for col in range_cols:
            df_fixed[col] = (df[col] / avg_price) * 100
    return df_fixed

sample_row = overall_df[overall_df['Timeframe'] == '1D'].iloc[0]
PERCENTAGES_NEED_FIX = sample_row['above_c2_high_pct'] > 100
RANGES_ARE_POINTS = sample_row['Avg_C2_Range'] > 10
overall_df = fix_dataframe(overall_df, PERCENTAGES_NEED_FIX, RANGES_ARE_POINTS)

print("✓ Data loaded and fixed\n")

# Define the three scenarios
scenarios = [
    {
        'pattern': 'above_high',
        'title': 'C2 CLOSES ABOVE C1 HIGH',
        'subtitle': 'What does C3 do?',
        'filename': 'SCENARIO_1_C2_Above_C1_High'
    },
    {
        'pattern': 'below_low',
        'title': 'C2 CLOSES BELOW C1 LOW',
        'subtitle': 'What does C3 do?',
        'filename': 'SCENARIO_2_C2_Below_C1_Low'
    },
    {
        'pattern': 'inside_range',
        'title': 'C2 CLOSES INSIDE C1 RANGE',
        'subtitle': 'What does C3 do?',
        'filename': 'SCENARIO_3_C2_Inside_C1'
    }
]

timeframes = ['15m', '30m', '1h', '4h', '8h', '1D']

# Create one comprehensive page for each scenario
for scenario in scenarios:
    print(f"Creating page: {scenario['title']}...")
    
    # Get data for this pattern
    pattern_data = overall_df[overall_df['Pattern'] == scenario['pattern']].copy()
    pattern_data = pattern_data.set_index('Timeframe').reindex(timeframes)
    
    # Create figure
    fig = plt.figure(figsize=(18, 14))
    fig.patch.set_facecolor('white')
    
    # Main title
    fig.text(0.5, 0.98, scenario['title'], ha='center', fontsize=16, 
             fontweight='bold', family='monospace')
    fig.text(0.5, 0.96, scenario['subtitle'], ha='center', fontsize=12, 
             family='monospace')
    
    # ========================================================================
    # SECTION 1: THREE OUTCOMES
    # ========================================================================
    
    fig.text(0.05, 0.88, 'THREE OUTCOMES (sum to 100%)', fontsize=11, 
             fontweight='bold', family='monospace')
    
    ax1 = plt.subplot(4, 1, 1)
    ax1.set_position([0.05, 0.78, 0.9, 0.08])
    ax1.axis('tight')
    ax1.axis('off')
    
    outcomes_data = []
    for tf in timeframes:
        if tf not in pattern_data.index:
            continue
        
        row = pattern_data.loc[tf]
        
        if scenario['pattern'] == 'above_high':
            cont = row['above_c2_high_pct']
            rev = row['below_c2_low_pct']
        elif scenario['pattern'] == 'below_low':
            cont = row['below_c2_low_pct']
            rev = row['above_c2_high_pct']
        else:
            cont = row['above_c2_high_pct']
            rev = row['below_c2_low_pct']
        
        neutral = 100 - cont - rev
        
        outcomes_data.append([
            tf,
            f"{int(row['Total_Count']):,}",
            f"{cont:.1f}%",
            f"{neutral:.1f}%",
            f"{rev:.1f}%"
        ])
    
    outcomes_headers = ['TF', 'N', 'Continue', 'Neutral', 'Reverse']
    
    outcomes_table = ax1.table(cellText=outcomes_data, colLabels=outcomes_headers,
                               cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
    outcomes_table.auto_set_font_size(False)
    outcomes_table.set_fontsize(9)
    outcomes_table.scale(1, 2.5)
    
    # Monochrome styling
    for i in range(len(outcomes_headers)):
        cell = outcomes_table[(0, i)]
        cell.set_facecolor('black')
        cell.set_text_props(weight='bold', color='white', size=9, family='monospace')
        cell.set_edgecolor('black')
        cell.set_linewidth(1)
    
    for i in range(1, len(outcomes_data) + 1):
        for j in range(len(outcomes_headers)):
            cell = outcomes_table[(i, j)]
            cell.set_facecolor('white')
            cell.set_text_props(size=9, family='monospace')
            cell.set_edgecolor('black')
            cell.set_linewidth(0.5)
    
    # ========================================================================
    # SECTION 2: LEVELS TOUCHED
    # ========================================================================
    
    fig.text(0.05, 0.64, 'LEVELS TOUCHED (includes wicks)', fontsize=11, 
             fontweight='bold', family='monospace')
    
    ax2 = plt.subplot(4, 1, 2)
    ax2.set_position([0.05, 0.54, 0.9, 0.08])
    ax2.axis('tight')
    ax2.axis('off')
    
    touched_data = []
    for tf in timeframes:
        if tf not in pattern_data.index:
            continue
        
        row = pattern_data.loc[tf]
        
        touched_data.append([
            tf,
            f"{row['above_c2_high_pct']:.1f}%",
            f"{row['above_c2_75pct_pct']:.1f}%",
            f"{row['above_c2_50pct_pct']:.1f}%",
            f"{row['above_c2_25pct_pct']:.1f}%",
            f"{row['below_c2_25pct_pct']:.1f}%",
            f"{row['below_c2_50pct_pct']:.1f}%",
            f"{row['below_c2_75pct_pct']:.1f}%",
            f"{row['below_c2_low_pct']:.1f}%"
        ])
    
    touched_headers = ['TF', '>High', '>75%', '>50%', '>25%', '<25%', '<50%', '<75%', '<Low']
    
    touched_table = ax2.table(cellText=touched_data, colLabels=touched_headers,
                              cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
    touched_table.auto_set_font_size(False)
    touched_table.set_fontsize(9)
    touched_table.scale(1, 2.5)
    
    for i in range(len(touched_headers)):
        cell = touched_table[(0, i)]
        cell.set_facecolor('black')
        cell.set_text_props(weight='bold', color='white', size=9, family='monospace')
        cell.set_edgecolor('black')
        cell.set_linewidth(1)
    
    for i in range(1, len(touched_data) + 1):
        for j in range(len(touched_headers)):
            cell = touched_table[(i, j)]
            cell.set_facecolor('white')
            cell.set_text_props(size=9, family='monospace')
            cell.set_edgecolor('black')
            cell.set_linewidth(0.5)
    
    # ========================================================================
    # SECTION 3: WHERE CLOSED
    # ========================================================================
    
    fig.text(0.05, 0.40, 'WHERE CLOSED (actual position)', fontsize=11, 
             fontweight='bold', family='monospace')
    
    ax3 = plt.subplot(4, 1, 3)
    ax3.set_position([0.05, 0.30, 0.9, 0.08])
    ax3.axis('tight')
    ax3.axis('off')
    
    closed_data = []
    for tf in timeframes:
        if tf not in pattern_data.index:
            continue
        
        row = pattern_data.loc[tf]
        
        closed_data.append([
            tf,
            f"{row['above_c2_high_pct']:.1f}%",
            f"{row['above_c2_close_pct']:.1f}%",
            f"{row['above_c2_open_pct']:.1f}%",
            f"{row['below_c2_open_pct']:.1f}%",
            f"{row['below_c2_close_pct']:.1f}%",
            f"{row['below_c2_low_pct']:.1f}%"
        ])
    
    closed_headers = ['TF', 'Cl>High', 'Cl>Close', 'Cl>Open', 'Cl<Open', 'Cl<Close', 'Cl<Low']
    
    closed_table = ax3.table(cellText=closed_data, colLabels=closed_headers,
                            cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
    closed_table.auto_set_font_size(False)
    closed_table.set_fontsize(9)
    closed_table.scale(1, 2.5)
    
    for i in range(len(closed_headers)):
        cell = closed_table[(0, i)]
        cell.set_facecolor('black')
        cell.set_text_props(weight='bold', color='white', size=9, family='monospace')
        cell.set_edgecolor('black')
        cell.set_linewidth(1)
    
    for i in range(1, len(closed_data) + 1):
        for j in range(len(closed_headers)):
            cell = closed_table[(i, j)]
            cell.set_facecolor('white')
            cell.set_text_props(size=9, family='monospace')
            cell.set_edgecolor('black')
            cell.set_linewidth(0.5)
    
    # ========================================================================
    # SECTION 4: CANDLE RANGES
    # ========================================================================
    
    fig.text(0.05, 0.16, 'CANDLE RANGES (% of price)', fontsize=11, 
             fontweight='bold', family='monospace')
    
    ax4 = plt.subplot(4, 1, 4)
    ax4.set_position([0.05, 0.06, 0.9, 0.08])
    ax4.axis('tight')
    ax4.axis('off')
    
    stats_data = []
    for tf in timeframes:
        if tf not in pattern_data.index:
            continue
        
        row = pattern_data.loc[tf]
        
        stats_data.append([
            tf,
            f"{row['Avg_C1_Range']:.3f}%",
            f"{row['Avg_C2_Range']:.3f}%",
            f"{row['Avg_C3_Range']:.3f}%"
        ])
    
    stats_headers = ['TF', 'Avg C1', 'Avg C2', 'Avg C3']
    
    stats_table = ax4.table(cellText=stats_data, colLabels=stats_headers,
                           cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
    stats_table.auto_set_font_size(False)
    stats_table.set_fontsize(9)
    stats_table.scale(1, 2.5)
    
    for i in range(len(stats_headers)):
        cell = stats_table[(0, i)]
        cell.set_facecolor('black')
        cell.set_text_props(weight='bold', color='white', size=9, family='monospace')
        cell.set_edgecolor('black')
        cell.set_linewidth(1)
    
    for i in range(1, len(stats_data) + 1):
        for j in range(len(stats_headers)):
            cell = stats_table[(i, j)]
            cell.set_facecolor('white')
            cell.set_text_props(size=9, family='monospace')
            cell.set_edgecolor('black')
            cell.set_linewidth(0.5)
    
    # Footer
    footer = f"NQ Futures 2013-2025 | TF=Timeframe | N=Sample Size | Cl=Closed"
    fig.text(0.5, 0.01, footer, ha='center', fontsize=8, 
             family='monospace', style='italic')
    
    plt.savefig(f'{base_path}\\{scenario["filename"]}.png', dpi=300, 
                bbox_inches='tight', facecolor='white', edgecolor='none')
    print(f"✓ Saved: {scenario['filename']}.png")
    plt.close()

print("\n" + "=" * 120)
print("MINIMALIST SUMMARIES COMPLETE!")
print("=" * 120)
print("\nGenerated 3 clean, monochrome one-page summaries:")
print("  • SCENARIO_1_C2_Above_C1_High.png")
print("  • SCENARIO_2_C2_Below_C1_Low.png")
print("  • SCENARIO_3_C2_Inside_C1.png")
print(f"\nAll files saved to: {base_path}")
print("\nDesign: Minimalist monochrome with monospace font")

CREATING MINIMALIST ONE-PAGE SUMMARIES
✓ Data loaded and fixed

Creating page: C2 CLOSES ABOVE C1 HIGH...
✓ Saved: SCENARIO_1_C2_Above_C1_High.png
Creating page: C2 CLOSES BELOW C1 LOW...
✓ Saved: SCENARIO_2_C2_Below_C1_Low.png
Creating page: C2 CLOSES INSIDE C1 RANGE...
✓ Saved: SCENARIO_3_C2_Inside_C1.png

MINIMALIST SUMMARIES COMPLETE!

Generated 3 clean, monochrome one-page summaries:
  • SCENARIO_1_C2_Above_C1_High.png
  • SCENARIO_2_C2_Below_C1_Low.png
  • SCENARIO_3_C2_Inside_C1.png

All files saved to: C:\Users\tammm\NQ and ES analysis

Design: Minimalist monochrome with monospace font


In [3]:
#!/usr/bin/env python
# coding: utf-8

"""
3-CANDLE PATTERN ANALYSIS - WITH CANDLE STRUCTURE TRACKING
Analyzes candle 3 behavior based on candle 2's relationship to candle 1
NOW INCLUDES: C1+C2 direction combinations (Momentum vs Exhaustion patterns)

NEW FEATURE: Tracks whether C1 and C2 are bullish/bearish to identify:
- Momentum patterns (C1 Bull + C2 Bull)
- Exhaustion patterns (C1 Bull + C2 Bear)
- Reversal patterns (C1 Bear + C2 Bull)
- Unusual patterns (C1 Bear + C2 Bear when breaking C1 high)
"""

import pandas as pd
import numpy as np
from collections import defaultdict
from datetime import time

# Load the data
file_path = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
df = pd.read_csv(file_path)

# Combine Date and Time into a datetime column
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.set_index('Datetime', inplace=True)

# Filter data from 2013 to 2025
start_date = pd.to_datetime('2013-01-01')
end_date = pd.to_datetime('2025-12-31')
df = df.loc[start_date:end_date]

print("=" * 120)
print("3-CANDLE PATTERN ANALYSIS - WITH CANDLE STRUCTURE TRACKING")
print("=" * 120)
print(f"\nData loaded: {len(df)} rows from {df.index[0]} to {df.index[-1]}")
print("\nNEW FEATURE: Tracking C1+C2 direction combinations")
print("  • Momentum: C1 Bull + C2 Bull (or C1 Bear + C2 Bear)")
print("  • Exhaustion: C1 Bull + C2 Bear (or C1 Bear + C2 Bull)")
print("  • This should reveal if 'exhaustion' patterns reverse more often!")

# Define timeframes to analyze
TIMEFRAMES = {
    '15m': '15min',
    '30m': '30min', 
    '1h': '1H',
    '4h': '4H',
    '8h': '8H',
    '1D': '1D'
}

# Define filter categories
STRENGTH_BUCKETS = [0, 0.1, 0.25, 0.5, 1.0, 2.0, float('inf')]
STRENGTH_LABELS = ['0-0.1%', '0.1-0.25%', '0.25-0.5%', '0.5-1.0%', '1.0-2.0%', '2.0%+']

def resample_to_timeframe(df, timeframe):
    """Resample 1-minute data to specified timeframe"""
    resampled = df.resample(timeframe).agg({
        'Open': 'first',
        'High': 'max',
        'Low': 'min',
        'Close': 'last',
        'Volume': 'sum'
    }).dropna()
    return resampled

def calculate_candle_levels(candle):
    """Calculate key levels for a candle"""
    high = candle['High']
    low = candle['Low']
    open_price = candle['Open']
    close = candle['Close']
    
    range_size = high - low
    
    levels = {
        'high': high,
        'low': low,
        'open': open_price,
        'close': close,
        'range': range_size,
        '75_pct': low + (range_size * 0.75),
        '50_pct': low + (range_size * 0.50),
        '25_pct': low + (range_size * 0.25)
    }
    
    return levels

def classify_candle2_vs_candle1(c1_levels, c2_levels):
    """
    Classify candle 2's close relative to candle 1
    Returns: 'above_high', 'below_low', 'inside_range'
    """
    c2_close = c2_levels['close']
    c1_high = c1_levels['high']
    c1_low = c1_levels['low']
    
    if c2_close > c1_high:
        return 'above_high'
    elif c2_close < c1_low:
        return 'below_low'
    else:
        return 'inside_range'

def analyze_candle3_vs_candle2(c2_levels, c3_levels):
    """
    Analyze where candle 3 closes relative to all candle 2 levels
    Returns dict of boolean flags
    """
    c3_close = c3_levels['close']
    
    results = {
        'above_c2_high': c3_close > c2_levels['high'],
        'below_c2_low': c3_close < c2_levels['low'],
        'above_c2_open': c3_close > c2_levels['open'],
        'below_c2_open': c3_close < c2_levels['open'],
        'above_c2_close': c3_close > c2_levels['close'],
        'below_c2_close': c3_close < c2_levels['close'],
        'above_c2_75pct': c3_close > c2_levels['75_pct'],
        'below_c2_75pct': c3_close < c2_levels['75_pct'],
        'above_c2_50pct': c3_close > c2_levels['50_pct'],
        'below_c2_50pct': c3_close < c2_levels['50_pct'],
        'above_c2_25pct': c3_close > c2_levels['25_pct'],
        'below_c2_25pct': c3_close < c2_levels['25_pct'],
    }
    
    return results

def get_time_bucket(timestamp):
    """Categorize time of day into buckets"""
    hour = timestamp.hour
    
    if 18 <= hour <= 23:
        return 'Overnight'
    elif 0 <= hour < 2:
        return 'Overnight'
    elif 2 <= hour < 9:
        return 'PreMarket'
    elif 9 <= hour < 16:
        return 'RTH'
    else:
        return 'After_Hours'

def get_day_of_week(timestamp):
    """Get day of week name"""
    days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    return days[timestamp.weekday()]

def get_candle_direction(candle):
    """Determine if candle is bullish or bearish"""
    return 'Bullish' if candle['Close'] > candle['Open'] else 'Bearish'

def calculate_breakout_strength_pct(c1_levels, c2_levels, pattern_type):
    """
    Calculate how far C2 closes beyond C1 as a percentage of C1's close price
    Returns percentage value
    """
    c1_close = c1_levels['close']
    c2_close = c2_levels['close']
    
    if pattern_type == 'above_high':
        c1_reference = c1_levels['high']
        distance = c2_close - c1_reference
    elif pattern_type == 'below_low':
        c1_reference = c1_levels['low']
        distance = c1_reference - c2_close
    else:
        return 0
    
    strength_pct = (distance / c1_close) * 100
    return strength_pct

def get_strength_bucket(strength_pct):
    """Categorize strength into buckets"""
    for i in range(len(STRENGTH_BUCKETS) - 1):
        if strength_pct >= STRENGTH_BUCKETS[i] and strength_pct < STRENGTH_BUCKETS[i+1]:
            return STRENGTH_LABELS[i]
    return STRENGTH_LABELS[-1]

def get_structure_type(c1_direction, c2_direction, pattern_type):
    """
    Classify the C1+C2 structure combination
    Returns: 'Momentum', 'Exhaustion', 'Reversal', or 'Unusual'
    """
    if pattern_type == 'above_high':
        # C2 closes above C1 high
        if c1_direction == 'Bullish' and c2_direction == 'Bullish':
            return 'Momentum'  # Both up
        elif c1_direction == 'Bullish' and c2_direction == 'Bearish':
            return 'Exhaustion'  # Up then sellers take over
        elif c1_direction == 'Bearish' and c2_direction == 'Bullish':
            return 'Reversal'  # Down then buyers reverse
        else:
            return 'Unusual'  # Both down but closes above high
    
    elif pattern_type == 'below_low':
        # C2 closes below C1 low
        if c1_direction == 'Bearish' and c2_direction == 'Bearish':
            return 'Momentum'  # Both down
        elif c1_direction == 'Bearish' and c2_direction == 'Bullish':
            return 'Exhaustion'  # Down then buyers take over
        elif c1_direction == 'Bullish' and c2_direction == 'Bearish':
            return 'Reversal'  # Up then sellers reverse
        else:
            return 'Unusual'  # Both up but closes below low
    
    else:
        # Inside range - structure matters less
        return f'{c1_direction}_{c2_direction}'

def analyze_timeframe(df_resampled, timeframe_name):
    """
    Analyze 3-candle patterns for a specific timeframe
    """
    print(f"\nAnalyzing {timeframe_name} timeframe...")
    
    # Initialize statistics storage with filters
    def create_stats_dict():
        return {
            'count': 0,
            'c3_outcomes': defaultdict(int),
            'c1_ranges': [],
            'c2_ranges': [],
            'c3_ranges': [],
            'c1_closes': [],
            'time_buckets': defaultdict(int),
            # Existing filters
            'by_c2_direction': {
                'Bullish': {'count': 0, 'outcomes': defaultdict(int)},
                'Bearish': {'count': 0, 'outcomes': defaultdict(int)}
            },
            'by_strength': {
                label: {'count': 0, 'outcomes': defaultdict(int)} 
                for label in STRENGTH_LABELS
            },
            'by_time_session': {
                session: {'count': 0, 'outcomes': defaultdict(int)}
                for session in ['RTH', 'PreMarket', 'Overnight', 'After_Hours']
            },
            'by_day_of_week': {
                day: {'count': 0, 'outcomes': defaultdict(int)}
                for day in ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
            },
            # NEW: Structure breakdown
            'by_structure': {
                'Momentum': {'count': 0, 'outcomes': defaultdict(int)},
                'Exhaustion': {'count': 0, 'outcomes': defaultdict(int)},
                'Reversal': {'count': 0, 'outcomes': defaultdict(int)},
                'Unusual': {'count': 0, 'outcomes': defaultdict(int)}
            },
            # NEW: Detailed C1+C2 combinations
            'by_c1_c2_combo': {
                'Bullish_Bullish': {'count': 0, 'outcomes': defaultdict(int)},
                'Bullish_Bearish': {'count': 0, 'outcomes': defaultdict(int)},
                'Bearish_Bullish': {'count': 0, 'outcomes': defaultdict(int)},
                'Bearish_Bearish': {'count': 0, 'outcomes': defaultdict(int)}
            }
        }
    
    stats = {
        'above_high': create_stats_dict(),
        'below_low': create_stats_dict(),
        'inside_range': create_stats_dict()
    }
    
    # Analyze consecutive 3-candle sequences
    for i in range(len(df_resampled) - 2):
        candle1 = df_resampled.iloc[i]
        candle2 = df_resampled.iloc[i + 1]
        candle3 = df_resampled.iloc[i + 2]
        
        # Skip weekends
        if candle2.name.weekday() >= 5:
            continue
        
        # Skip if any candle has zero range
        if candle1['High'] == candle1['Low'] or candle2['High'] == candle2['Low'] or candle3['High'] == candle3['Low']:
            continue
        
        # Calculate levels for each candle
        c1_levels = calculate_candle_levels(candle1)
        c2_levels = calculate_candle_levels(candle2)
        c3_levels = calculate_candle_levels(candle3)
        
        # Classify candle 2 vs candle 1
        c2_classification = classify_candle2_vs_candle1(c1_levels, c2_levels)
        
        # Analyze candle 3 vs candle 2
        c3_analysis = analyze_candle3_vs_candle2(c2_levels, c3_levels)
        
        # Get filter attributes
        time_bucket = get_time_bucket(candle2.name)
        day_of_week = get_day_of_week(candle2.name)
        c1_direction = get_candle_direction(candle1)  # NEW
        c2_direction = get_candle_direction(candle2)
        strength_pct = calculate_breakout_strength_pct(c1_levels, c2_levels, c2_classification)
        strength_bucket = get_strength_bucket(strength_pct) if c2_classification != 'inside_range' else None
        
        # NEW: Get structure type
        structure_type = get_structure_type(c1_direction, c2_direction, c2_classification)
        c1_c2_combo = f'{c1_direction}_{c2_direction}'
        
        # Update statistics
        pattern_stats = stats[c2_classification]
        pattern_stats['count'] += 1
        pattern_stats['c1_ranges'].append(c1_levels['range'])
        pattern_stats['c2_ranges'].append(c2_levels['range'])
        pattern_stats['c3_ranges'].append(c3_levels['range'])
        pattern_stats['c1_closes'].append(c1_levels['close'])
        pattern_stats['time_buckets'][time_bucket] += 1
        
        # Increment counts
        pattern_stats['by_c2_direction'][c2_direction]['count'] += 1
        pattern_stats['by_time_session'][time_bucket]['count'] += 1
        pattern_stats['by_day_of_week'][day_of_week]['count'] += 1
        if strength_bucket:
            pattern_stats['by_strength'][strength_bucket]['count'] += 1
        
        # NEW: Track structure combinations
        if c2_classification != 'inside_range':
            pattern_stats['by_structure'][structure_type]['count'] += 1
        pattern_stats['by_c1_c2_combo'][c1_c2_combo]['count'] += 1
        
        # Record all candle 3 outcomes
        for outcome, occurred in c3_analysis.items():
            if occurred:
                pattern_stats['c3_outcomes'][outcome] += 1
                pattern_stats['by_c2_direction'][c2_direction]['outcomes'][outcome] += 1
                pattern_stats['by_time_session'][time_bucket]['outcomes'][outcome] += 1
                pattern_stats['by_day_of_week'][day_of_week]['outcomes'][outcome] += 1
                
                if strength_bucket:
                    pattern_stats['by_strength'][strength_bucket]['outcomes'][outcome] += 1
                
                # NEW: Track structure outcomes
                if c2_classification != 'inside_range':
                    pattern_stats['by_structure'][structure_type]['outcomes'][outcome] += 1
                pattern_stats['by_c1_c2_combo'][c1_c2_combo]['outcomes'][outcome] += 1
    
    return stats

def display_results(all_timeframe_stats):
    """Display comprehensive results - abbreviated version"""
    
    print("\n" + "=" * 120)
    print("RESULTS: 3-CANDLE PATTERN ANALYSIS")
    print("=" * 120)
    print("\n(Full results in CSV exports - showing structure breakdown here)")
    
    for tf_name, stats in all_timeframe_stats.items():
        print(f"\n{'=' * 120}")
        print(f"TIMEFRAME: {tf_name}")
        print(f"{'=' * 120}")
        
        for pattern_type in ['above_high', 'below_low']:
            pattern_stats = stats[pattern_type]
            
            if pattern_stats['count'] == 0:
                continue
            
            print(f"\n{'-' * 120}")
            pattern_name = "ABOVE C1 HIGH" if pattern_type == 'above_high' else "BELOW C1 LOW"
            print(f"PATTERN: C2 closes {pattern_name}")
            print(f"{'-' * 120}")
            
            total_count = pattern_stats['count']
            print(f"Total: {total_count:,}")
            
            # NEW: Structure breakdown
            print(f"\nSTRUCTURE BREAKDOWN (C1+C2 Direction Combinations):")
            print(f"{'Type':15s} {'Count':>10s} {'%':>8s} {'Continue':>10s} {'Neutral':>10s} {'Reverse':>10s}")
            print("-" * 80)
            
            for struct_type in ['Momentum', 'Exhaustion', 'Reversal', 'Unusual']:
                struct_data = pattern_stats['by_structure'][struct_type]
                struct_count = struct_data['count']
                
                if struct_count == 0:
                    continue
                
                struct_pct = (struct_count / total_count) * 100
                
                # Calculate outcomes
                if pattern_type == 'above_high':
                    cont = struct_data['outcomes'].get('above_c2_high', 0)
                    rev = struct_data['outcomes'].get('below_c2_low', 0)
                else:
                    cont = struct_data['outcomes'].get('below_c2_low', 0)
                    rev = struct_data['outcomes'].get('above_c2_high', 0)
                
                neut = struct_count - cont - rev
                
                cont_pct = (cont / struct_count) * 100
                neut_pct = (neut / struct_count) * 100
                rev_pct = (rev / struct_count) * 100
                
                print(f"{struct_type:15s} {struct_count:10,} {struct_pct:7.1f}% "
                      f"{cont_pct:9.1f}% {neut_pct:9.1f}% {rev_pct:9.1f}%")

def export_to_csv(all_timeframe_stats):
    """Export results to CSV - includes new structure breakdowns"""
    
    # [Keep all existing exports 1-5 the same as before]
    
    # Export 6: NEW - Structure breakdown
    structure_export_data = []
    
    for tf_name, stats in all_timeframe_stats.items():
        for pattern_type in ['above_high', 'below_low']:
            pattern_stats = stats[pattern_type]
            
            if pattern_stats['count'] == 0:
                continue
            
            for struct_type in ['Momentum', 'Exhaustion', 'Reversal', 'Unusual']:
                struct_data = pattern_stats['by_structure'][struct_type]
                struct_total = struct_data['count']
                struct_outcomes = struct_data['outcomes']
                
                if struct_total == 0:
                    continue
                
                row = {
                    'Timeframe': tf_name,
                    'Pattern': pattern_type,
                    'Structure_Type': struct_type,
                    'Total_Count': struct_total
                }
                
                # Add all outcomes
                for outcome in ['above_c2_high', 'below_c2_low', 'above_c2_close', 'below_c2_close',
                               'above_c2_50pct', 'below_c2_50pct']:
                    count = struct_outcomes.get(outcome, 0)
                    pct = (count / struct_total) * 100 if struct_total > 0 else 0
                    row[f'{outcome}_pct'] = pct
                    row[f'{outcome}'] = count
                
                structure_export_data.append(row)
    
    structure_df = pd.DataFrame(structure_export_data)
    structure_output_path = r"C:\Users\tammm\NQ and ES analysis\three_candle_by_STRUCTURE.csv"
    structure_df.to_csv(structure_output_path, index=False)
    print(f"\n✓ NEW: Structure breakdown exported to: {structure_output_path}")
    
    # Export 7: NEW - Detailed C1+C2 combinations
    combo_export_data = []
    
    for tf_name, stats in all_timeframe_stats.items():
        for pattern_type in ['above_high', 'below_low', 'inside_range']:
            pattern_stats = stats[pattern_type]
            
            if pattern_stats['count'] == 0:
                continue
            
            for combo in ['Bullish_Bullish', 'Bullish_Bearish', 'Bearish_Bullish', 'Bearish_Bearish']:
                combo_data = pattern_stats['by_c1_c2_combo'][combo]
                combo_total = combo_data['count']
                combo_outcomes = combo_data['outcomes']
                
                if combo_total == 0:
                    continue
                
                c1_dir, c2_dir = combo.split('_')
                
                row = {
                    'Timeframe': tf_name,
                    'Pattern': pattern_type,
                    'C1_Direction': c1_dir,
                    'C2_Direction': c2_dir,
                    'Total_Count': combo_total
                }
                
                for outcome in ['above_c2_high', 'below_c2_low', 'above_c2_close', 'below_c2_close']:
                    count = combo_outcomes.get(outcome, 0)
                    pct = (count / combo_total) * 100 if combo_total > 0 else 0
                    row[f'{outcome}_pct'] = pct
                    row[f'{outcome}'] = count
                
                combo_export_data.append(row)
    
    combo_df = pd.DataFrame(combo_export_data)
    combo_output_path = r"C:\Users\tammm\NQ and ES analysis\three_candle_by_C1_C2_COMBO.csv"
    combo_df.to_csv(combo_output_path, index=False)
    print(f"✓ NEW: C1+C2 combinations exported to: {combo_output_path}")

# Main execution
print("\n" + "=" * 120)
print("PROCESSING MULTIPLE TIMEFRAMES")
print("=" * 120)

all_timeframe_stats = {}

for tf_name, tf_freq in TIMEFRAMES.items():
    df_resampled = resample_to_timeframe(df, tf_freq)
    print(f"\n{tf_name}: {len(df_resampled):,} candles created")
    
    stats = analyze_timeframe(df_resampled, tf_name)
    all_timeframe_stats[tf_name] = stats

# Display results
display_results(all_timeframe_stats)

# Export to CSV
print("\n" + "=" * 120)
print("EXPORTING RESULTS")
print("=" * 120)
export_to_csv(all_timeframe_stats)

print("\n" + "=" * 120)
print("ANALYSIS COMPLETE - WITH STRUCTURE TRACKING!")
print("=" * 120)
print(f"\nNEW EXPORTS:")
print(f"  • three_candle_by_STRUCTURE.csv - Momentum/Exhaustion/Reversal/Unusual breakdown")
print(f"  • three_candle_by_C1_C2_COMBO.csv - Full C1+C2 direction combinations")
print(f"\nNow you can test if 'Exhaustion' patterns really do reverse more often!")

3-CANDLE PATTERN ANALYSIS - WITH CANDLE STRUCTURE TRACKING

Data loaded: 4161727 rows from 2013-01-02 06:00:00 to 2025-03-14 16:59:00

NEW FEATURE: Tracking C1+C2 direction combinations
  • Momentum: C1 Bull + C2 Bull (or C1 Bear + C2 Bear)
  • Exhaustion: C1 Bull + C2 Bear (or C1 Bear + C2 Bull)
  • This should reveal if 'exhaustion' patterns reverse more often!

PROCESSING MULTIPLE TIMEFRAMES

15m: 285,181 candles created

Analyzing 15m timeframe...

30m: 143,991 candles created

Analyzing 30m timeframe...

1h: 72,371 candles created

Analyzing 1h timeframe...

4h: 19,416 candles created

Analyzing 4h timeframe...

8h: 10,027 candles created

Analyzing 8h timeframe...

1D: 3,783 candles created

Analyzing 1D timeframe...

RESULTS: 3-CANDLE PATTERN ANALYSIS

(Full results in CSV exports - showing structure breakdown here)

TIMEFRAME: 15m

------------------------------------------------------------------------------------------------------------------------
PATTERN: C2 closes ABOVE C1

In [4]:
#!/usr/bin/env python
# coding: utf-8

"""
3-CANDLE PATTERN ANALYSIS - COMPLETE WITH TIME-OF-DAY TRACKING

For ALL timeframes, tracks C1+C2 structure (Momentum/Exhaustion/Reversal/Unusual)
Plus time-of-day effects:
- 30m, 1h, 4h, 8h: Session (Asian/London/NY) + Hour (0-23)
- 1D: Day of week (Monday-Friday)

This reveals if patterns behave differently at different times!
"""

import pandas as pd
import numpy as np
from collections import defaultdict

# Load the data
file_path = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
df = pd.read_csv(file_path)

# Combine Date and Time into a datetime column
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.set_index('Datetime', inplace=True)

# Filter data from 2013 to 2025
start_date = pd.to_datetime('2013-01-01')
end_date = pd.to_datetime('2025-12-31')
df = df.loc[start_date:end_date]

print("=" * 120)
print("3-CANDLE PATTERN ANALYSIS - WITH TIME-OF-DAY TRACKING")
print("=" * 120)
print(f"\nData loaded: {len(df)} rows from {df.index[0]} to {df.index[-1]}")
print("\nTRACKING:")
print("  • C1+C2 structure (Momentum/Exhaustion/Reversal/Unusual)")
print("  • For 30m/1h/4h/8h: Session + Hour when C2 closes")
print("  • For 1D: Day of week when C2 closes")
print("  • Do patterns behave differently at different times?")

# Define timeframes to analyze
TIMEFRAMES = {
    '30m': '30min', 
    '1h': '1H',
    '4h': '4H',
    '8h': '8H',
    '1D': '1D'
}

def get_trading_session(timestamp):
    """
    Determine trading session (EST timezone)
    Asian: 20:00-02:00 | London: 02:00-08:00
    NY_Futures: 08:00-09:30 & 16:00-17:00 | NY_Cash: 09:30-16:00
    """
    hour = timestamp.hour
    minute = timestamp.minute
    
    if hour >= 20 or hour < 2:
        return 'Asian'
    elif 2 <= hour < 8:
        return 'London'
    elif (hour == 9 and minute >= 30) or (10 <= hour < 16):
        return 'NY_Cash'
    elif (8 <= hour < 9) or (hour == 9 and minute < 30) or (hour == 16):
        return 'NY_Futures'
    else:
        return 'Afterhours'

def get_day_of_week(timestamp):
    """Get day of week name"""
    days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    return days[timestamp.weekday()]

def resample_to_timeframe(df, timeframe):
    """Resample 1-minute data to specified timeframe"""
    resampled = df.resample(timeframe).agg({
        'Open': 'first',
        'High': 'max',
        'Low': 'min',
        'Close': 'last',
        'Volume': 'sum'
    }).dropna()
    return resampled

def calculate_candle_levels(candle):
    """Calculate key levels for a candle"""
    high = candle['High']
    low = candle['Low']
    open_price = candle['Open']
    close = candle['Close']
    
    range_size = high - low
    
    levels = {
        'high': high,
        'low': low,
        'open': open_price,
        'close': close,
        'range': range_size,
        '75_pct': low + (range_size * 0.75),
        '50_pct': low + (range_size * 0.50),
        '25_pct': low + (range_size * 0.25)
    }
    
    return levels

def classify_candle2_vs_candle1(c1_levels, c2_levels):
    """Classify candle 2's close relative to candle 1"""
    c2_close = c2_levels['close']
    c1_high = c1_levels['high']
    c1_low = c1_levels['low']
    
    if c2_close > c1_high:
        return 'above_high'
    elif c2_close < c1_low:
        return 'below_low'
    else:
        return 'inside_range'

def analyze_candle3_vs_candle2(c2_levels, c3_levels):
    """Analyze where candle 3 closes relative to candle 2 levels"""
    c3_close = c3_levels['close']
    
    results = {
        'above_c2_high': c3_close > c2_levels['high'],
        'below_c2_low': c3_close < c2_levels['low'],
        'above_c2_open': c3_close > c2_levels['open'],
        'below_c2_open': c3_close < c2_levels['open'],
        'above_c2_close': c3_close > c2_levels['close'],
        'below_c2_close': c3_close < c2_levels['close'],
        'above_c2_75pct': c3_close > c2_levels['75_pct'],
        'below_c2_75pct': c3_close < c2_levels['75_pct'],
        'above_c2_50pct': c3_close > c2_levels['50_pct'],
        'below_c2_50pct': c3_close < c2_levels['50_pct'],
        'above_c2_25pct': c3_close > c2_levels['25_pct'],
        'below_c2_25pct': c3_close < c2_levels['25_pct'],
    }
    
    return results

def get_candle_direction(candle):
    """Determine if candle is bullish or bearish"""
    return 'Bullish' if candle['Close'] > candle['Open'] else 'Bearish'

def get_structure_type(c1_direction, c2_direction, pattern_type):
    """
    Classify C1+C2 structure combination
    Returns: 'Momentum', 'Exhaustion', 'Reversal', or 'Unusual'
    """
    if pattern_type == 'above_high':
        if c1_direction == 'Bullish' and c2_direction == 'Bullish':
            return 'Momentum'
        elif c1_direction == 'Bullish' and c2_direction == 'Bearish':
            return 'Exhaustion'
        elif c1_direction == 'Bearish' and c2_direction == 'Bullish':
            return 'Reversal'
        else:
            return 'Unusual'
    elif pattern_type == 'below_low':
        if c1_direction == 'Bearish' and c2_direction == 'Bearish':
            return 'Momentum'
        elif c1_direction == 'Bearish' and c2_direction == 'Bullish':
            return 'Exhaustion'
        elif c1_direction == 'Bullish' and c2_direction == 'Bearish':
            return 'Reversal'
        else:
            return 'Unusual'
    else:
        return f'{c1_direction}_{c2_direction}'

def analyze_timeframe(df_resampled, timeframe_name):
    """Analyze 3-candle patterns with time-of-day tracking"""
    print(f"\nAnalyzing {timeframe_name} timeframe...")
    
    # Initialize statistics storage
    def create_stats_dict():
        return {
            'count': 0,
            'c3_outcomes': defaultdict(int),
            'c1_ranges': [],
            'c2_ranges': [],
            'c3_ranges': [],
            'c1_closes': [],
            # Structure breakdown
            'by_structure': {
                'Momentum': {'count': 0, 'outcomes': defaultdict(int)},
                'Exhaustion': {'count': 0, 'outcomes': defaultdict(int)},
                'Reversal': {'count': 0, 'outcomes': defaultdict(int)},
                'Unusual': {'count': 0, 'outcomes': defaultdict(int)}
            },
            # NEW: Session breakdown (for intraday timeframes)
            'by_session': {
                'Asian': {'count': 0, 'outcomes': defaultdict(int), 'by_structure': defaultdict(int)},
                'London': {'count': 0, 'outcomes': defaultdict(int), 'by_structure': defaultdict(int)},
                'NY_Futures': {'count': 0, 'outcomes': defaultdict(int), 'by_structure': defaultdict(int)},
                'NY_Cash': {'count': 0, 'outcomes': defaultdict(int), 'by_structure': defaultdict(int)},
                'Afterhours': {'count': 0, 'outcomes': defaultdict(int), 'by_structure': defaultdict(int)}
            },
            # NEW: Hour breakdown (for intraday timeframes)
            'by_hour': {hour: {'count': 0, 'outcomes': defaultdict(int), 'by_structure': defaultdict(int)} 
                       for hour in range(24)},
            # NEW: Day of week breakdown (for daily timeframe)
            'by_day_of_week': {
                'Monday': {'count': 0, 'outcomes': defaultdict(int), 'by_structure': defaultdict(int)},
                'Tuesday': {'count': 0, 'outcomes': defaultdict(int), 'by_structure': defaultdict(int)},
                'Wednesday': {'count': 0, 'outcomes': defaultdict(int), 'by_structure': defaultdict(int)},
                'Thursday': {'count': 0, 'outcomes': defaultdict(int), 'by_structure': defaultdict(int)},
                'Friday': {'count': 0, 'outcomes': defaultdict(int), 'by_structure': defaultdict(int)}
            }
        }
    
    stats = {
        'above_high': create_stats_dict(),
        'below_low': create_stats_dict(),
        'inside_range': create_stats_dict()
    }
    
    # Analyze consecutive 3-candle sequences
    for i in range(len(df_resampled) - 2):
        candle1 = df_resampled.iloc[i]
        candle2 = df_resampled.iloc[i + 1]
        candle3 = df_resampled.iloc[i + 2]
        
        # Skip weekends
        if candle2.name.weekday() >= 5:
            continue
        
        # Skip if any candle has zero range
        if candle1['High'] == candle1['Low'] or candle2['High'] == candle2['Low'] or candle3['High'] == candle3['Low']:
            continue
        
        # Calculate levels
        c1_levels = calculate_candle_levels(candle1)
        c2_levels = calculate_candle_levels(candle2)
        c3_levels = calculate_candle_levels(candle3)
        
        # Classify pattern
        c2_classification = classify_candle2_vs_candle1(c1_levels, c2_levels)
        c3_analysis = analyze_candle3_vs_candle2(c2_levels, c3_levels)
        
        # Get C1 and C2 directions
        c1_direction = get_candle_direction(candle1)
        c2_direction = get_candle_direction(candle2)
        structure_type = get_structure_type(c1_direction, c2_direction, c2_classification)
        
        # Get time attributes based on when C2 closes
        c2_timestamp = candle2.name
        
        if timeframe_name == '1D':
            time_attribute = get_day_of_week(c2_timestamp)
            time_type = 'day_of_week'
        else:
            session = get_trading_session(c2_timestamp)
            hour = c2_timestamp.hour
            time_type = 'session_hour'
        
        # Update statistics
        pattern_stats = stats[c2_classification]
        pattern_stats['count'] += 1
        pattern_stats['c1_ranges'].append(c1_levels['range'])
        pattern_stats['c2_ranges'].append(c2_levels['range'])
        pattern_stats['c3_ranges'].append(c3_levels['range'])
        pattern_stats['c1_closes'].append(c1_levels['close'])
        
        # Update structure stats
        if c2_classification != 'inside_range':
            pattern_stats['by_structure'][structure_type]['count'] += 1
        
        # Update time-of-day stats
        if timeframe_name == '1D':
            pattern_stats['by_day_of_week'][time_attribute]['count'] += 1
            if c2_classification != 'inside_range':
                pattern_stats['by_day_of_week'][time_attribute]['by_structure'][structure_type] += 1
        else:
            pattern_stats['by_session'][session]['count'] += 1
            pattern_stats['by_hour'][hour]['count'] += 1
            if c2_classification != 'inside_range':
                pattern_stats['by_session'][session]['by_structure'][structure_type] += 1
                pattern_stats['by_hour'][hour]['by_structure'][structure_type] += 1
        
        # Record outcomes
        for outcome, occurred in c3_analysis.items():
            if occurred:
                pattern_stats['c3_outcomes'][outcome] += 1
                
                if c2_classification != 'inside_range':
                    pattern_stats['by_structure'][structure_type]['outcomes'][outcome] += 1
                
                if timeframe_name == '1D':
                    pattern_stats['by_day_of_week'][time_attribute]['outcomes'][outcome] += 1
                else:
                    pattern_stats['by_session'][session]['outcomes'][outcome] += 1
                    pattern_stats['by_hour'][hour]['outcomes'][outcome] += 1
    
    return stats

def export_to_csv(all_timeframe_stats):
    """Export results including time-of-day breakdowns"""
    
    # Export 1: Overall results (same as before)
    export_data = []
    
    for tf_name, stats in all_timeframe_stats.items():
        for pattern_type in ['above_high', 'below_low', 'inside_range']:
            pattern_stats = stats[pattern_type]
            
            if pattern_stats['count'] == 0:
                continue
            
            total_count = pattern_stats['count']
            avg_c1_close = np.mean(pattern_stats['c1_closes'])
            avg_c1_range = np.mean(pattern_stats['c1_ranges'])
            avg_c2_range = np.mean(pattern_stats['c2_ranges'])
            avg_c3_range = np.mean(pattern_stats['c3_ranges'])
            
            avg_c1_range_pct = (avg_c1_range / avg_c1_close) * 100
            avg_c2_range_pct = (avg_c2_range / avg_c1_close) * 100
            avg_c3_range_pct = (avg_c3_range / avg_c1_close) * 100
            
            row = {
                'Timeframe': tf_name,
                'Pattern': pattern_type,
                'Total_Count': total_count,
                'Avg_C1_Range': avg_c1_range_pct,
                'Avg_C2_Range': avg_c2_range_pct,
                'Avg_C3_Range': avg_c3_range_pct,
            }
            
            for outcome in ['above_c2_high', 'below_c2_low', 'above_c2_open', 'below_c2_open',
                           'above_c2_close', 'below_c2_close', 'above_c2_75pct', 'below_c2_75pct',
                           'above_c2_50pct', 'below_c2_50pct', 'above_c2_25pct', 'below_c2_25pct']:
                count = pattern_stats['c3_outcomes'].get(outcome, 0)
                pct = (count / total_count) * 100
                row[f'{outcome}_pct'] = pct
                row[f'{outcome}'] = count
            
            export_data.append(row)
    
    export_df = pd.DataFrame(export_data)
    output_path = r"C:\Users\tammm\NQ and ES analysis\three_candle_WITH_TIME_overall.csv"
    export_df.to_csv(output_path, index=False)
    print(f"\n✓ Overall results: {output_path}")
    
    # Export 2: Structure breakdown
    structure_export_data = []
    
    for tf_name, stats in all_timeframe_stats.items():
        for pattern_type in ['above_high', 'below_low']:
            pattern_stats = stats[pattern_type]
            
            if pattern_stats['count'] == 0:
                continue
            
            for struct_type in ['Momentum', 'Exhaustion', 'Reversal', 'Unusual']:
                struct_data = pattern_stats['by_structure'][struct_type]
                struct_total = struct_data['count']
                struct_outcomes = struct_data['outcomes']
                
                if struct_total == 0:
                    continue
                
                row = {
                    'Timeframe': tf_name,
                    'Pattern': pattern_type,
                    'Structure_Type': struct_type,
                    'Total_Count': struct_total
                }
                
                for outcome in ['above_c2_high', 'below_c2_low', 'above_c2_close', 'below_c2_close',
                               'above_c2_50pct', 'below_c2_50pct']:
                    count = struct_outcomes.get(outcome, 0)
                    pct = (count / struct_total) * 100 if struct_total > 0 else 0
                    row[f'{outcome}_pct'] = pct
                    row[f'{outcome}'] = count
                
                structure_export_data.append(row)
    
    structure_df = pd.DataFrame(structure_export_data)
    structure_output_path = r"C:\Users\tammm\NQ and ES analysis\three_candle_WITH_TIME_by_structure.csv"
    structure_df.to_csv(structure_output_path, index=False)
    print(f"✓ Structure breakdown: {structure_output_path}")
    
    # Export 3: NEW - Session breakdown (for intraday timeframes)
    session_export_data = []
    
    for tf_name, stats in all_timeframe_stats.items():
        if tf_name == '1D':
            continue  # Skip daily for session analysis
        
        for pattern_type in ['above_high', 'below_low']:
            pattern_stats = stats[pattern_type]
            
            if pattern_stats['count'] == 0:
                continue
            
            for session in ['Asian', 'London', 'NY_Futures', 'NY_Cash']:
                session_data = pattern_stats['by_session'][session]
                session_total = session_data['count']
                session_outcomes = session_data['outcomes']
                
                if session_total == 0:
                    continue
                
                # Overall session stats
                row = {
                    'Timeframe': tf_name,
                    'Pattern': pattern_type,
                    'Session': session,
                    'Total_Count': session_total
                }
                
                for outcome in ['above_c2_high', 'below_c2_low', 'above_c2_close', 'below_c2_close']:
                    count = session_outcomes.get(outcome, 0)
                    pct = (count / session_total) * 100 if session_total > 0 else 0
                    row[f'{outcome}_pct'] = pct
                    row[f'{outcome}'] = count
                
                # Add structure breakdown counts
                for struct_type in ['Momentum', 'Exhaustion', 'Reversal', 'Unusual']:
                    count = session_data['by_structure'][struct_type]
                    pct = (count / session_total) * 100 if session_total > 0 else 0
                    row[f'{struct_type}_count'] = count
                    row[f'{struct_type}_pct'] = pct
                
                session_export_data.append(row)
    
    session_df = pd.DataFrame(session_export_data)
    session_output_path = r"C:\Users\tammm\NQ and ES analysis\three_candle_WITH_TIME_by_session.csv"
    session_df.to_csv(session_output_path, index=False)
    print(f"✓ Session breakdown: {session_output_path}")
    
    # Export 4: NEW - Hour breakdown (for intraday timeframes)
    hour_export_data = []
    
    for tf_name, stats in all_timeframe_stats.items():
        if tf_name == '1D':
            continue
        
        for pattern_type in ['above_high', 'below_low']:
            pattern_stats = stats[pattern_type]
            
            if pattern_stats['count'] == 0:
                continue
            
            for hour in range(24):
                hour_data = pattern_stats['by_hour'][hour]
                hour_total = hour_data['count']
                hour_outcomes = hour_data['outcomes']
                
                if hour_total == 0:
                    continue
                
                row = {
                    'Timeframe': tf_name,
                    'Pattern': pattern_type,
                    'Hour': hour,
                    'Total_Count': hour_total
                }
                
                for outcome in ['above_c2_high', 'below_c2_low']:
                    count = hour_outcomes.get(outcome, 0)
                    pct = (count / hour_total) * 100 if hour_total > 0 else 0
                    row[f'{outcome}_pct'] = pct
                    row[f'{outcome}'] = count
                
                # Add structure breakdown
                for struct_type in ['Momentum', 'Exhaustion']:
                    count = hour_data['by_structure'][struct_type]
                    pct = (count / hour_total) * 100 if hour_total > 0 else 0
                    row[f'{struct_type}_count'] = count
                    row[f'{struct_type}_pct'] = pct
                
                hour_export_data.append(row)
    
    hour_df = pd.DataFrame(hour_export_data)
    hour_output_path = r"C:\Users\tammm\NQ and ES analysis\three_candle_WITH_TIME_by_hour.csv"
    hour_df.to_csv(hour_output_path, index=False)
    print(f"✓ Hour breakdown: {hour_output_path}")
    
    # Export 5: NEW - Day of week breakdown (for daily timeframe)
    dow_export_data = []
    
    for tf_name, stats in all_timeframe_stats.items():
        if tf_name != '1D':
            continue
        
        for pattern_type in ['above_high', 'below_low']:
            pattern_stats = stats[pattern_type]
            
            if pattern_stats['count'] == 0:
                continue
            
            for day in ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']:
                day_data = pattern_stats['by_day_of_week'][day]
                day_total = day_data['count']
                day_outcomes = day_data['outcomes']
                
                if day_total == 0:
                    continue
                
                row = {
                    'Timeframe': tf_name,
                    'Pattern': pattern_type,
                    'Day_Of_Week': day,
                    'Total_Count': day_total
                }
                
                for outcome in ['above_c2_high', 'below_c2_low', 'above_c2_close', 'below_c2_close']:
                    count = day_outcomes.get(outcome, 0)
                    pct = (count / day_total) * 100 if day_total > 0 else 0
                    row[f'{outcome}_pct'] = pct
                    row[f'{outcome}'] = count
                
                # Add structure breakdown
                for struct_type in ['Momentum', 'Exhaustion', 'Reversal', 'Unusual']:
                    count = day_data['by_structure'][struct_type]
                    pct = (count / day_total) * 100 if day_total > 0 else 0
                    row[f'{struct_type}_count'] = count
                    row[f'{struct_type}_pct'] = pct
                
                dow_export_data.append(row)
    
    dow_df = pd.DataFrame(dow_export_data)
    dow_output_path = r"C:\Users\tammm\NQ and ES analysis\three_candle_WITH_TIME_by_day_of_week.csv"
    dow_df.to_csv(dow_output_path, index=False)
    print(f"✓ Day of week breakdown: {dow_output_path}")

# Main execution
print("\n" + "=" * 120)
print("PROCESSING TIMEFRAMES")
print("=" * 120)

all_timeframe_stats = {}

for tf_name, tf_freq in TIMEFRAMES.items():
    df_resampled = resample_to_timeframe(df, tf_freq)
    print(f"\n{tf_name}: {len(df_resampled):,} candles created")
    
    stats = analyze_timeframe(df_resampled, tf_name)
    all_timeframe_stats[tf_name] = stats

# Export to CSV
print("\n" + "=" * 120)
print("EXPORTING RESULTS")
print("=" * 120)
export_to_csv(all_timeframe_stats)

print("\n" + "=" * 120)
print("ANALYSIS COMPLETE!")
print("=" * 120)
print("\nGenerated 5 CSV files:")
print("  1. three_candle_WITH_TIME_overall.csv - Overall statistics")
print("  2. three_candle_WITH_TIME_by_structure.csv - Momentum/Exhaustion breakdown")
print("  3. three_candle_WITH_TIME_by_session.csv - Asian/London/NY breakdown (30m-8h)")
print("  4. three_candle_WITH_TIME_by_hour.csv - Hour 0-23 breakdown (30m-8h)")
print("  5. three_candle_WITH_TIME_by_day_of_week.csv - Mon-Fri breakdown (1D)")
print("\nNow you can see if:")
print("  • Momentum patterns in Asian session behave differently than NY")
print("  • Certain hours have better continuation rates")
print("  • Specific days of week are more reliable")

3-CANDLE PATTERN ANALYSIS - WITH TIME-OF-DAY TRACKING

Data loaded: 4161727 rows from 2013-01-02 06:00:00 to 2025-03-14 16:59:00

TRACKING:
  • C1+C2 structure (Momentum/Exhaustion/Reversal/Unusual)
  • For 30m/1h/4h/8h: Session + Hour when C2 closes
  • For 1D: Day of week when C2 closes
  • Do patterns behave differently at different times?

PROCESSING TIMEFRAMES

30m: 143,991 candles created

Analyzing 30m timeframe...

1h: 72,371 candles created

Analyzing 1h timeframe...

4h: 19,416 candles created

Analyzing 4h timeframe...

8h: 10,027 candles created

Analyzing 8h timeframe...

1D: 3,783 candles created

Analyzing 1D timeframe...

EXPORTING RESULTS

✓ Overall results: C:\Users\tammm\NQ and ES analysis\three_candle_WITH_TIME_overall.csv
✓ Structure breakdown: C:\Users\tammm\NQ and ES analysis\three_candle_WITH_TIME_by_structure.csv
✓ Session breakdown: C:\Users\tammm\NQ and ES analysis\three_candle_WITH_TIME_by_session.csv
✓ Hour breakdown: C:\Users\tammm\NQ and ES analysis\three